# 02 Events and predictors
One table of events (parliamentary elections, crises, planned cabinet changes, the two ambiguous rows) with day 0 and
day +1 on each index's trading calendar, eligibility, overlaps and the political predictors. It also writes the
political dates of every country (used in 03 to clean estimation windows and normal days) and the sample flow. Prices
are used only for the trading calendar. ParlGov tables are joined on ids, never on names.

In [ ]:
import sys
from pathlib import Path

# Folder of helpers.py: the notebook's own folder or, on Nuvolos (where the kernel starts elsewhere), found by search.
# The search accepts only a code/ folder of THIS project (helpers.py, analysis.py and ../data/crises.csv) and stops
# if it finds more than one, so another project's helpers.py can never be imported by mistake.
def find_code_dir():
    found = set()
    for root in [Path.cwd(), Path("/files"), Path.home()]:
        for p in [root / "helpers.py", *root.glob("**/code/helpers.py")]:
            d = p.parent
            if p.exists() and (d / "analysis.py").exists() and (d.parent / "data" / "crises.csv").exists():
                found.add(d.resolve())
        if found:
            break
    if len(found) != 1:
        raise FileNotFoundError(f"expected one code/ folder of this project, found {sorted(map(str, found))}")
    return found.pop()


sys.path.insert(0, str(find_code_dir()))

import numpy as np
import pandas as pd

from helpers import (DATA, MAJORITY_SHARE, MIN_HISTORY, TICKERS, candidate_cabinets, load_cabinets, load_elections,
                     load_prices, raw_file, save_derived)
from analysis import (ANALYSIS_GROUPS, CRISIS_LABELS, apply_robustness_set, calendar_position, in_invalid_span,
                      load_robustness_sets, overlap_flags, valid_closes)

## Inputs and checks
`crises.csv` must list exactly the ParlGov cabinets of the 23 countries that started since 1997 without a new
election. Its label and date columns are read as text, so "1", "0", "ambiguous" and empty stay as coded.

In [ ]:
DATE_COLS = ["parlgov_start_date", "trigger_date", "alt_trigger_dates", "resignation_date"]
crises = pd.read_csv(DATA / "crises.csv", dtype={c: str for c in ["is_crisis", *DATE_COLS]})
crises[["is_crisis", *DATE_COLS]] = crises[["is_crisis", *DATE_COLS]].fillna("")
rob, elections, cab = load_robustness_sets(), load_elections(), load_cabinets()
el_party = pd.read_csv(raw_file("parlgov_view_election"), parse_dates=["election_date"])
el_party = el_party[(el_party["election_type"] == "parliament") & el_party["country_name_short"].isin(list(TICKERS))]
cab1 = cab.drop_duplicates("cabinet_id").set_index("cabinet_id")

ids, coded = crises["parlgov_cabinet_id"], crises["is_crisis"] != ""
label_counts = crises["is_crisis"].value_counts().to_dict()
assert crises["event_id"].is_unique and set(candidate_cabinets(cab)["cabinet_id"]) == set(ids)
assert (crises["event_id"] == crises["country"] + "_" + ids.astype(str)).all()
assert (cab1.loc[ids, "start_date"].dt.strftime("%Y-%m-%d").to_numpy() == crises["parlgov_start_date"]).all()
assert label_counts == {"1": 80, "0": 44, "ambiguous": 2, "": 23}, label_counts
assert (crises.loc[coded, "trigger_date"] != "").all()
assert len(elections) == 174 and not el_party.duplicated(["election_id", "party_id"]).any()
assert rob["event_id"].isin(crises["event_id"]).all() and set(rob["action"]) <= {"drop", "relabel"}
by_id = crises.set_index("event_id")     # an alternative day 0 of a robustness set is a date already coded for that row
assert all(set(r["new_trigger_date"].split(";")) <= set(by_id.loc[r["event_id"], DATE_COLS].str.split(";").explode())
           for _, r in rob[rob["new_trigger_date"] != ""].iterrows())
print("crises.csv:", label_counts, "| elections since 1997:", len(elections))

## Events
- Elections: the incumbent is the cabinet in office at the vote (`previous_cabinet_id` of the election).
- Cabinet changes coded "1" (crisis), "0" (planned) or "ambiguous"; uncoded rows are not events. The incumbent is the
  cabinet that fell (`previous_cabinet_id` of the new cabinet); the new cabinet does not exist on day 0.
- One extra row (variant = set name) for each alternative day 0 of a robustness set; with two listed dates, day 0
  comes from the first.

In [ ]:
cr = crises[coded]
events = pd.concat([
    pd.DataFrame({"event_id": elections["country_name_short"] + "_E" + elections["election_id"].astype(str),
                  "country": elections["country_name_short"], "group": "election", "event_type": "",
                  "source_date": elections["election_date"], "incumbent": elections["previous_cabinet_id"],
                  "new_parliament": elections["election_id"]}),
    pd.DataFrame({"event_id": cr["event_id"], "country": cr["country"],
                  "group": cr["is_crisis"].map({**CRISIS_LABELS, "ambiguous": "ambiguous"}), "event_type": cr["event_type"],
                  "source_date": pd.to_datetime(cr["trigger_date"]),
                  "incumbent": cr["parlgov_cabinet_id"].map(cab1["previous_cabinet_id"]), "new_parliament": np.nan}),
], ignore_index=True).assign(variant="main")
alt = [events[events["event_id"] == r["event_id"]].assign(variant=r["set"], source_date=pd.Timestamp(r["new_trigger_date"].split(";")[0]))
       for _, r in rob[rob["new_trigger_date"] != ""].iterrows()]
events = pd.concat([events, *alt], ignore_index=True)
assert not events.duplicated(["event_id", "variant"]).any() and events["incumbent"].isin(cab1.index).all()
print(events.groupby(["variant", "group"]).size().to_string())

## Day 0 and eligibility
The trading calendar of a country is its set of valid closes (`analysis.valid_closes`, also used by 03). Not valid
are closes inside a span of `INVALID_CLOSES` (the Yahoo Greek series is another series from 19 Dec 2014 to 12 May
2015: it matches the official Athens Exchange closes to the cent outside that span only) and closes repeated inside a
run of at least `STALE_ZERO_RUN` zero returns.

Day 0 is the first trading day on or after the trigger date or the election date; day +1 is the next one. An event is
eligible if day 0 has at least `MIN_HISTORY` earlier trading days, day +1 exists, and the period from source date to
day +1 does not touch invalid closes (this excludes only the Greek election of January 2015).

In [ ]:
prices = load_prices()
calendars = {c: valid_closes(prices, c).index for c in TICKERS}
print("Closes removed (invalid span or stale prices):",
      {c: n for c in TICKERS if (n := int(prices[TICKERS[c]].notna().sum()) - len(calendars[c]))})


def day(c, pos):
    """Trading day at position pos of country c's calendar (NaT if pos is missing or past the end)."""
    return calendars[c][int(pos)] if pos < len(calendars[c]) else pd.NaT


events["day0_pos"] = [calendar_position(calendars[c], [d])[0] for c, d in zip(events["country"], events["source_date"])]
events["day0"] = pd.to_datetime([day(c, p) for c, p in zip(events["country"], events["day0_pos"])])
events["day1"] = pd.to_datetime([day(c, p + 1) for c, p in zip(events["country"], events["day0_pos"])])
end = events["day1"].fillna(events["day0"]).fillna(events["source_date"])
bad_data = np.array([in_invalid_span(c, [s], [e])[0] for c, s, e in zip(events["country"], events["source_date"], end)])
has_history = events["day0_pos"] >= MIN_HISTORY
events["eligible"] = has_history & events["day1"].notna() & ~bad_data
events["ineligible_reason"] = np.select(
    [events["day0"].isna(), bad_data, ~has_history, events["day1"].isna()],
    ["no trading day on or after the source date", "index data invalid between source date and day +1",
     f"fewer than {MIN_HISTORY} earlier trading days", "no day +1 in the price file"], default="")
assert (events["day0"].dropna() >= events.loc[events["day0"].notna(), "source_date"]).all()
assert (n_el := int(events.loc[events["group"] == "election", "eligible"].sum())) == 144, n_el
print(events.loc[~events["eligible"]].groupby(["group", "ineligible_reason"]).size().to_string())

## Political dates
Every date of `crises.csv` (trigger, alternative trigger, resignation and ParlGov start dates, coded rows or not) and
every parliamentary election in ParlGov (also before 1997, since the first estimation windows reach back into 1996),
each mapped to its first trading day as day 0 is. A date inside invalid closes or after the last close has no
trading day.

In [ ]:
pol = pd.concat([crises[["country", "event_id", c]].rename(columns={c: "date"}).assign(source=c) for c in DATE_COLS])
pol = pol.assign(date=pol["date"].str.split(";")).explode("date").query("date != ''").assign(date=lambda d: pd.to_datetime(d["date"]))
el_all = el_party.drop_duplicates("election_id")
pol = pd.concat([pol, pd.DataFrame({"country": el_all["country_name_short"], "date": el_all["election_date"], "source": "election_date",
                                    "event_id": el_all["country_name_short"] + "_E" + el_all["election_id"].astype(str)})],
                ignore_index=True)
pol["trading_day"] = pd.to_datetime([pd.NaT if in_invalid_span(c, [d])[0] else day(c, calendar_position(calendars[c], [d])[0])
                                     for c, d in zip(pol["country"], pol["date"])])
political_dates = pol[["country", "date", "source", "event_id", "trading_day"]].sort_values(
    ["country", "date", "source", "event_id"]).reset_index(drop=True)
assert set(zip(events["country"], events["source_date"])) <= set(zip(pol["country"], pol["date"]))
print(len(political_dates), "political dates;", int(political_dates["trading_day"].isna().sum()), "without a trading day")

## Predictors known on day 0
- `coalition`: the incumbent has two or more cabinet parties. `minority`: their seats are below `MAJORITY_SHARE` of
  `election_seats_total`. A technocratic incumbent (no seats held by cabinet parties) counts as a minority
  single-party cabinet.
- `enp`: effective number of parties, 1 / sum of squared seat shares, of the parliament in office on day 0 (the
  incumbent's parliament; for an election, the outgoing one). Seat shares use the seats of the parties listed for
  that election.
- `majority_crisis`: 1 for a crisis in which the government lost its majority or a partner left the coalition
  (`LOST_MAJORITY`, `COALITION_EXIT`), 0 otherwise. With `crisis` in the model, it compares these crises with
  leadership crises (`LEADERSHIP`, `ELECTORAL_DEFEAT`).

Two quantities of elections are outcomes of the vote, known only with the result: `enp_new` (of the new parliament)
and `lr_shift`, the absolute change of the seat-weighted mean `left_right` between the outgoing and the new parliament
(parties without a value are left out of the mean). They are used only descriptively.

In [ ]:
listed = el_party[el_party["seats"] > 0]
enp = 1 / ((listed["seats"] / listed.groupby("election_id")["seats"].transform("sum")) ** 2).groupby(listed["election_id"]).sum()
known = listed[listed["left_right"].notna()]
lr = (known["seats"] * known["left_right"]).groupby(known["election_id"]).sum() / known.groupby("election_id")["seats"].sum()

cp = cab[cab["cabinet_party"] == 1].groupby("cabinet_id")
gov_seats = cp["seats"].sum().reindex(cab1.index, fill_value=0)
seats_known = cp["seats"].apply(lambda s: s.notna().all()).reindex(cab1.index, fill_value=True)
techno = gov_seats.eq(0) & seats_known
minority = (gov_seats / cab1["election_seats_total"] < MAJORITY_SHARE).where(seats_known & cab1["election_seats_total"].gt(0)).mask(techno, True)
coalition = cp["party_id"].nunique().reindex(cab1.index, fill_value=0).ge(2) & ~techno

inc, is_el = events["incumbent"], events["group"] == "election"
parliament = inc.map(cab1["election_id"])
events = events.assign(crisis=(events["group"] == "crisis").astype(int),
                       majority_crisis=((events["group"] == "crisis") & events["event_type"].isin(["LOST_MAJORITY", "COALITION_EXIT"])).astype(int),
                       coalition=inc.map(coalition).astype(float), minority=inc.map(minority).astype(float),
                       enp=parliament.map(enp), enp_new=events["new_parliament"].map(enp),
                       lr_shift=(events["new_parliament"].map(lr) - parliament.map(lr)).abs())
feats = ["coalition", "minority", "enp"]
assert (elections["previous_parliament_election_id"].to_numpy() == parliament[is_el].to_numpy()).all()
assert (inc.map(cab1["start_date"]) < events["source_date"]).all()           # incumbent in office before day 0
assert events[feats].notna().all().all() and events.loc[is_el, ["enp_new", "lr_shift"]].notna().all().all()
print(events[events["variant"] == "main"].groupby("group")[["majority_crisis", *feats, "enp_new", "lr_shift"]].mean().round(2).to_string())

## Overlaps and robustness sets
Among eligible analysis events of the same country (ambiguous rows are not analysed), day 0 at most `OVERLAP_DROP`
trading days apart: the earlier is kept and the later dropped. `apply_robustness_set` repeats this for each set,
since its drops, relabels and new day 0s change which events overlap.

In [ ]:
main = (events["variant"] == "main") & events["group"].isin(ANALYSIS_GROUPS)
events["overlap_drop"] = overlap_flags(events[main]).reindex(events.index)
by_set = {s: apply_robustness_set(events, s) for s in ["main", *dict.fromkeys(rob["set"])]}
n_crises = {s: int((t["group"] == "crisis").sum()) for s, t in by_set.items()}
assert n_crises == {"main": 80, "R_add": 88, "R_drop": 78, "R_vol": 73, "R_bundle": 82, "R_tv": 64, "R_split": 79}, n_crises
print("Overlap drops:", {s: t.loc[t["overlap_drop"], "event_id"].tolist() for s, t in by_set.items()})

## Outputs
`events.csv` (one row per event and variant), `political_dates.csv` and `sample_flow.csv` (main labels: ParlGov
candidates -> coded -> eligible -> after overlap, by group).

In [ ]:
t = by_set["main"]
flow = pd.DataFrame([("candidates", "election", len(elections)), ("candidates", "cabinet change", len(crises)),
                     ("coded", "ambiguous, not analysed", int((cr["is_crisis"] == "ambiguous").sum())),
                     ("coded", "uncoded", int((~coded).sum()))]
                    + [(stage, g, int(((t["group"] == g) & rows).sum()))
                       for stage, rows in [("coded", True), ("eligible", t["eligible"]), ("after overlap", t["eligible"] & ~t["overlap_drop"])]
                       for g in ANALYSIS_GROUPS], columns=["stage", "group", "n"])
print(flow.to_string(index=False))
cols = ["event_id", "variant", "country", "group", "event_type", "source_date", "day0", "day1", "day0_pos", "eligible",
        "ineligible_reason", "overlap_drop", "crisis", "majority_crisis", "coalition", "minority", "enp", "enp_new", "lr_shift"]
for name, df in [("events", events[cols].sort_values(["country", "source_date", "event_id", "variant"])),
                 ("political_dates", political_dates), ("sample_flow", flow)]:
    print(save_derived(df, name), len(df), "rows")